In [2]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

# **Detalle Ordenes**

In [150]:
url_clientes=r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\1.raw\detalle_ordenes.csv'
df_detalleordene=pd.read_csv(url_clientes,sep=',')
print(f'Cantidad de filas {df_detalleordene.shape[0]} y columnas {df_detalleordene.shape[1]}')

Cantidad de filas 20321 y columnas 6


In [60]:
import pandas as pd
import json
import os
from datetime import datetime


def validar_y_procesar_csv(ruta_archivo, configuracion):
    nombre = os.path.basename(ruta_archivo)

    reporte = {
        "archivo": nombre,
        "fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "estado": "exitoso",
        "errores": []
    }

    # 1. Lectura
    try:
        df = pd.read_csv(ruta_archivo)
        print(
            f"✅ Archivo '{nombre}' cargado. "
            f"{df.shape[0]} filas, {df.shape[1]} columnas."
        )
    except Exception as e:
        return {
            "archivo": nombre,
            "fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "estado": "error_lectura",
            "mensaje": str(e)
        }

    # 2. Validación de columnas
    cols_actuales = set(df.columns)
    cols_esperadas = set(configuracion['columnas'])

    faltantes = cols_esperadas - cols_actuales
    sobrantes = cols_actuales - cols_esperadas

    if faltantes or sobrantes:
        reporte["estado"] = "error_estructura"
        reporte["errores"].append({
            "tipo": "columnas",
            "faltantes": list(faltantes),
            "sobrantes": list(sobrantes)
        })

    # 3. Validación de tipos y reglas de negocio
    if reporte["estado"] == "exitoso":

        # Validación de tipos
        for col, tipo in configuracion['tipos'].items():

            if col not in df.columns:
                continue

            tipo_actual = str(df[col].dtype)

            if tipo == 'str':
                valido = tipo_actual == 'str'
            else:
                valido = tipo_actual == tipo

            if not valido:
                reporte["estado"] = "error_tipos"
                reporte["errores"].append({
                    "tipo": "tipo_dato",
                    "columna": col,
                    "esperado": tipo,
                    "actual": tipo_actual
                })

        # Reglas de negocio
        for col, regla in configuracion['reglas'].items():

            if col not in df.columns:
                continue

            if regla == "estricto":
                invalidos = df[df[col] <= 0]

            elif regla == "minimo_cero":
                invalidos = df[df[col] < 0]

            else:
                continue

            if not invalidos.empty:
                reporte["estado"] = "error_negocio"
                reporte["errores"].append({
                    "tipo": "regla_negocio",
                    "columna": col,
                    "cantidad_invalidos": len(invalidos)
                })

    # 4. Guardar reporte JSON
    ruta_salida = (
        r"D:\Github-Time\Solutions-to-Coding-Challenges"
        r"\Data_Backus\data\2.processed"
    )

    os.makedirs(ruta_salida, exist_ok=True)

    fecha_archivo = datetime.now().strftime("%Y%m%d_%H%M%S")

    nombre_json = (
        f"reporte_{nombre.replace('.csv', '')}_{fecha_archivo}.json"
    )

    ruta_json = os.path.join(ruta_salida, nombre_json)

    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(
            reporte,
            f,
            indent=4,
            ensure_ascii=False
        )

    print(f"📄 Reporte guardado en: {ruta_json}")

    return reporte


# ============================================
# CONFIGURACIÓN DEL CLIENTE
# ============================================

configuracion_cliente = {
    'columnas': [
        'orden_id',
        'linea',
        'producto_id',
        'cantidad_unidades',
        'precio_unitario',
        'descuento_pct'
    ],
    'tipos': {
        'orden_id': 'str',
        'linea': 'int64',
        'producto_id': 'str',
        'cantidad_unidades': 'int64',
        'precio_unitario': 'float64',
        'descuento_pct': 'int64'
    },
    'reglas': {
        'cantidad_unidades': 'estricto',
        'precio_unitario': 'estricto',
        'descuento_pct': 'minimo_cero',
        'linea': 'estricto'
    }
}


# ============================================
# EJECUCIÓN
# ============================================

resultado = validar_y_procesar_csv(
    r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\1.raw\clientes.csv',
    configuracion_cliente
)

print("\nResultado:")
print(json.dumps(resultado, indent=4, ensure_ascii=False))

✅ Archivo 'clientes.csv' cargado. 20321 filas, 6 columnas.
📄 Reporte guardado en: D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\2.processed\reporte_clientes_20260620_152032.json

Resultado:
{
    "archivo": "clientes.csv",
    "fecha_ejecucion": "2026-06-20 15:20:32",
    "estado": "error_negocio",
    "errores": [
        {
            "tipo": "regla_negocio",
            "columna": "cantidad_unidades",
            "cantidad_invalidos": 278
        },
        {
            "tipo": "regla_negocio",
            "columna": "precio_unitario",
            "cantidad_invalidos": 429
        }
    ]
}


In [48]:
# 1. Definimos la máscara (la condición lógica)
# Es mejor usar "~" (operador NOT) para extraer lo que NO cumple
condicion_limpia = (
    (df_cliente['cantidad_unidades'] > 0) & 
    (df_cliente['precio_unitario'] > 0) & 
    (df_cliente['descuento_pct'] >= 0) & 
    (df_cliente['linea'] > 0)
)

# 2. Creamos el set de datos limpio (los que SÍ cumplen)
df_limpio = df_cliente[condicion_limpia].copy()

# 3. Creamos el set de datos de errores (los que NO cumplen)
# Usamos el símbolo '~' para invertir la condición
df_sucios = df_cliente[~condicion_limpia].copy()

# 4. Reporte rápido para tu consola
print(f"✅ Registros limpios: {len(df_limpio)}")
print(f"❌ Registros con errores: {len(df_sucios)}")

✅ Registros limpios: 19622
❌ Registros con errores: 699


In [50]:
import os

# 1. Definir la carpeta de destino
output_dir = r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\2.processed'

# Asegurarse de que la carpeta exista (por si acaso)
os.makedirs(output_dir, exist_ok=True)

# 2. Definir las rutas completas
path_clean = os.path.join(output_dir, 'clients_cleaned.csv')
path_dirty = os.path.join(output_dir, 'clients_dirty.csv')

# 3. Guardar los archivos
df_limpio.to_csv(path_clean, index=False)
df_sucios.to_csv(path_dirty, index=False)

print(f"✅ Archivos guardados exitosamente:")
print(f" - Limpio: {path_clean}")
print(f" - Sucios: {path_dirty}")

✅ Archivos guardados exitosamente:
 - Limpio: D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\2.processed\clients_cleaned.csv
 - Sucios: D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\2.processed\clients_dirty.csv


# **Ordenes**

In [152]:
url_detalleordenes=r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\1.raw\ordenes.csv'
df_ordenes=pd.read_csv(url_detalleordenes,sep=',')
print(f'Cantidad de filas {df_ordenes.shape[0]} y columnas {df_ordenes.shape[1]}')

Cantidad de filas 5787 y columnas 8


In [ ]:
url_detalleordenes=r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\1.raw\detalle_ordenes.csv'
df_detordenes=pd.read_csv(url_detalleordenes,sep=',')
print(f'Cantidad de filas {df_detordenes.shape[0]} y columnas {df_detordenes.shape[1]}')

def validar_esquema_y_tipos(df):
    """
    Valida la estructura y tipos de datos del DataFrame.
    Retorna un resumen de éxito/fallo por cada columna.
    """
    # 1. Definir los tipos esperados
    tipos_esperados = {
        'orden_id': ['object', 'str'],
        'cliente_id': ['object', 'str'],
        'estado': ['object', 'str'],
        'monto_total': ['float64', 'float32']
    }
    
    columnas_fecha = ['fecha_pedido', 'fecha_entrega', 'fecha_proceso']
    
    reporte_validacion = {"cumplen": [], "no_cumplen": []}
    
    # 2. Validación de tipos de texto y numéricos
    for col, tipos in tipos_esperados.items():
        if col in df.columns and str(df[col].dtype) in tipos:
            reporte_validacion["cumplen"].append(col)
        else:
            reporte_validacion["no_cumplen"].append(col)
            
    # 3. Validación de fechas (se espera que ya sean datetime o convertibles)
    for col in columnas_fecha:
        if col in df.columns and 'datetime' in str(df[col].dtype):
            reporte_validacion["cumplen"].append(col)
        else:
            reporte_validacion["no_cumplen"].append(col)
            
    return reporte_validacion

# --- Ejecución ---
resumen = validar_esquema_y_tipos(df_detordenes)

print("📊 Resumen de validación de esquemas:")
print(f"✅ Columnas correctas: {resumen['cumplen']}")
print(f"❌ Columnas con tipo incorrecto: {resumen['no_cumplen']}")

if len(resumen['no_cumplen']) > 0:
    print("⚠️ Acción requerida: Debe convertir las columnas listadas en rojo antes de proceder.")
else:
    print("🎉 ¡Todo listo! El esquema es correcto.")
    
    
def verificar_columnas_requeridas(df, columnas_requeridas):
    """
    Verifica que las columnas esperadas existan en el DataFrame.
    """
    columnas_actuales = set(df.columns)
    columnas_esperadas = set(columnas_requeridas)
    
    faltantes = columnas_esperadas - columnas_actuales
    sobrantes = columnas_actuales - columnas_esperadas
    
    if faltantes:
        raise KeyError(f"❌ Error: Faltan las siguientes columnas obligatorias: {faltantes}")
    
    if sobrantes:
        print(f"⚠️ Aviso: Se encontraron columnas extra que no están en el esquema: {sobrantes}")
    
    print("✅ Estructura validada: Todas las columnas requeridas están presentes.")
    return True

# --- Uso en tu flujo ---

columnas_obligatorias = [
    'orden_id', 'cliente_id', 'fecha_pedido', 'fecha_entrega', 
    'fecha_proceso', 'canal_venta', 'estado', 'monto_total'
]

# Ejecutamos la validación inmediatamente después de leer el CSV
try:
    verificar_columnas_requeridas(df_detordenes, columnas_obligatorias)
except KeyError as e:
    print(e)
    
    
def validar_longitud_fija(df, columna, nombre_regla):
    """
    Valida que todos los valores en la columna tengan la misma longitud.
    """
    # Convertimos a string y calculamos longitudes
    longitudes = df[columna].astype(str).str.len()
    
    min_len = longitudes.min()
    max_len = longitudes.max()
    
    if min_len != max_len:
        print(f"❌ Error en '{columna}': La longitud no es fija. Mín: {min_len}, Máx: {max_len}")
        
        # Identificamos las filas que no tienen la longitud "dominante" (la más común)
        longitud_correcta = longitudes.mode()[0]
        registros_erroneos = df[longitudes != longitud_correcta]
        
        print(f"   Se encontraron {len(registros_erroneos)} registros con longitud anómala.")
        return False, registros_erroneos
    else:
        print(f"✅ Validación de '{columna}' exitosa: Todos los registros tienen longitud {min_len}.")
        return True, None

# --- Aplicación a tus columnas ---

# Validamos orden_id
es_valido_orden, errores_orden = validar_longitud_fija(df_detordenes, 'orden_id', 'Longitud Orden ID')

# Validamos cliente_id
es_valido_cliente, errores_cliente = validar_longitud_fija(df_detordenes, 'cliente_id', 'Longitud Cliente ID')


import pandas as pd

def procesar_fechas_multiples(df, lista_columnas):
    """
    Convierte múltiples columnas a datetime, preservando originales 
    y reportando errores de formato.
    """
    reporte_calidad = {}
    
    for col in lista_columnas:
        if col in df.columns:
            # 1. Respaldo de seguridad para auditoría
            df[f'{col}_original'] = df[col]
            
            # 2. Limpieza inteligente:
            # Intento 1: Formato estándar YYYY-MM-DD
            fechas = pd.to_datetime(df[col], errors='coerce', format='%Y-%m-%d')
            
            # Intento 2: Formato europeo DD/MM/YYYY solo en los fallidos
            mask_nat = fechas.isna()
            if mask_nat.any():
                fechas.loc[mask_nat] = pd.to_datetime(
                    df.loc[mask_nat, col], 
                    errors='coerce', 
                    format='%d/%m/%Y'
                )
            
            # Asignar la columna procesada
            df[col] = fechas
            
            # 3. Registrar cuántos fallaron para el JSON de reporte
            errores = int(df[col].isna().sum())
            reporte_calidad[col] = errores
            
    return df, reporte_calidad

# --- Cómo usarla en tu flujo ---
columnas_fechas = ['fecha_pedido', 'fecha_entrega', 'fecha_proceso']

# Ejecución
df_detordenes, reporte_fechas = procesar_fechas_multiples(df_detordenes, columnas_fechas)

print("✅ Fechas procesadas. Resumen de errores por columna:")
print(reporte_fechas)



def estandarizar_estados(df, columna, estados_validos):
    # 1. Limpieza básica: quitar espacios y convertir a mayúsculas
    df[columna] = df[columna].str.strip().str.upper()
    
    # 2. Reemplazo: Si el valor no está en la lista de estados_validos, lo cambia a "OTROS"
    # Usamos .apply() para una lógica personalizada
    df[columna] = df[columna].apply(lambda x: x if x in estados_validos else "OTROS")
    
    return df

# --- Uso en tu flujo ---

# Definimos los estados que sí son válidos para tu negocio
lista_estados_permitidos = ['ENTREGADO', 'ANULADO', 'PENDIENTE', 'DEVUELTO']

# Aplicamos la función
df_detordenes = estandarizar_estados(df_detordenes, 'estado', lista_estados_permitidos)

# Verificamos
print(df_detordenes['estado'].value_counts())


def validar_montos(df):
    """
    Valida que todos los montos sean mayores a 0.
    Retorna el df filtrado y un reporte de errores.
    """
    # Identificar inválidos
    invalidos = df[df['monto_total'] <= 0].copy()
    
    # Filtrar válidos
    df_limpio = df[df['monto_total'] > 0].copy()
    
    # Registrar reporte
    reporte_error = {
        "cantidad_invalidos": len(invalidos),
        "promedio_monto_invalido": float(invalidos['monto_total'].mean()) if not invalidos.empty else 0
    }
    
    return df_limpio, invalidos, reporte_error

# --- Integración ---
df_limpio, df_sucios_monto, info_montos = validar_montos(df_detordenes)

if info_montos["cantidad_invalidos"] > 0:
    print(f"⚠️ Alerta: Se detectaron {info_montos['cantidad_invalidos']} registros con monto <= 0.")

Cantidad de filas 5787 y columnas 8
📊 Resumen de validación de esquemas:
✅ Columnas correctas: ['orden_id', 'cliente_id', 'estado', 'monto_total']
❌ Columnas con tipo incorrecto: ['fecha_pedido', 'fecha_entrega', 'fecha_proceso']
⚠️ Acción requerida: Debe convertir las columnas listadas en rojo antes de proceder.
✅ Estructura validada: Todas las columnas requeridas están presentes.
✅ Validación de 'orden_id' exitosa: Todos los registros tienen longitud 10.
✅ Validación de 'cliente_id' exitosa: Todos los registros tienen longitud 6.
✅ Fechas procesadas. Resumen de errores por columna:
{'fecha_pedido': 0, 'fecha_entrega': 0, 'fecha_proceso': 0}
estado
ENTREGADO    5094
ANULADO       284
PENDIENTE     238
DEVUELTO      171
Name: count, dtype: int64
⚠️ Alerta: Se detectaron 38 registros con monto <= 0.


In [137]:
def validar_esquema_y_tipos(df):
    """
    Valida la estructura y tipos de datos del DataFrame.
    Retorna un resumen de éxito/fallo por cada columna.
    """
    # 1. Definir los tipos esperados
    tipos_esperados = {
        'orden_id': ['object', 'str'],
        'cliente_id': ['object', 'str'],
        'estado': ['object', 'str'],
        'monto_total': ['float64', 'float32']
    }
    
    columnas_fecha = ['fecha_pedido', 'fecha_entrega', 'fecha_proceso']
    
    reporte_validacion = {"cumplen": [], "no_cumplen": []}
    
    # 2. Validación de tipos de texto y numéricos
    for col, tipos in tipos_esperados.items():
        if col in df.columns and str(df[col].dtype) in tipos:
            reporte_validacion["cumplen"].append(col)
        else:
            reporte_validacion["no_cumplen"].append(col)
            
    # 3. Validación de fechas (se espera que ya sean datetime o convertibles)
    for col in columnas_fecha:
        if col in df.columns and 'datetime' in str(df[col].dtype):
            reporte_validacion["cumplen"].append(col)
        else:
            reporte_validacion["no_cumplen"].append(col)
            
    return reporte_validacion

# --- Ejecución ---
resumen = validar_esquema_y_tipos(df_detordenes)

print("📊 Resumen de validación de esquemas:")
print(f"✅ Columnas correctas: {resumen['cumplen']}")
print(f"❌ Columnas con tipo incorrecto: {resumen['no_cumplen']}")

if len(resumen['no_cumplen']) > 0:
    print("⚠️ Acción requerida: Debe convertir las columnas listadas en rojo antes de proceder.")
else:
    print("🎉 ¡Todo listo! El esquema es correcto.")

📊 Resumen de validación de esquemas:
✅ Columnas correctas: ['orden_id', 'cliente_id', 'estado', 'monto_total']
❌ Columnas con tipo incorrecto: ['fecha_pedido', 'fecha_entrega', 'fecha_proceso']
⚠️ Acción requerida: Debe convertir las columnas listadas en rojo antes de proceder.


In [138]:
url_detalleordenes=r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\1.raw\detalle_ordenes.csv'
df_detordenes=pd.read_csv(url_detalleordenes,sep=',')
print(f'Cantidad de filas {df_detordenes.shape[0]} y columnas {df_detordenes.shape[1]}')

def validar_esquema_y_tipos(df):
    """
    Valida la estructura y tipos de datos del DataFrame.
    Retorna un resumen de éxito/fallo por cada columna.
    """
    # 1. Definir los tipos esperados
    tipos_esperados = {
        'orden_id': ['object', 'str'],
        'cliente_id': ['object', 'str'],
        'estado': ['object', 'str'],
        'monto_total': ['float64', 'float32']
    }
    
    columnas_fecha = ['fecha_pedido', 'fecha_entrega', 'fecha_proceso']
    
    reporte_validacion = {"cumplen": [], "no_cumplen": []}
    
    # 2. Validación de tipos de texto y numéricos
    for col, tipos in tipos_esperados.items():
        if col in df.columns and str(df[col].dtype) in tipos:
            reporte_validacion["cumplen"].append(col)
        else:
            reporte_validacion["no_cumplen"].append(col)
            
    # 3. Validación de fechas (se espera que ya sean datetime o convertibles)
    for col in columnas_fecha:
        if col in df.columns and 'datetime' in str(df[col].dtype):
            reporte_validacion["cumplen"].append(col)
        else:
            reporte_validacion["no_cumplen"].append(col)
            
    return reporte_validacion

# --- Ejecución ---
resumen = validar_esquema_y_tipos(df_detordenes)

print("📊 Resumen de validación de esquemas:")
print(f"✅ Columnas correctas: {resumen['cumplen']}")
print(f"❌ Columnas con tipo incorrecto: {resumen['no_cumplen']}")

if len(resumen['no_cumplen']) > 0:
    print("⚠️ Acción requerida: Debe convertir las columnas listadas en rojo antes de proceder.")
else:
    print("🎉 ¡Todo listo! El esquema es correcto.")
    
    
def verificar_columnas_requeridas(df, columnas_requeridas):
    """
    Verifica que las columnas esperadas existan en el DataFrame.
    """
    columnas_actuales = set(df.columns)
    columnas_esperadas = set(columnas_requeridas)
    
    faltantes = columnas_esperadas - columnas_actuales
    sobrantes = columnas_actuales - columnas_esperadas
    
    if faltantes:
        raise KeyError(f"❌ Error: Faltan las siguientes columnas obligatorias: {faltantes}")
    
    if sobrantes:
        print(f"⚠️ Aviso: Se encontraron columnas extra que no están en el esquema: {sobrantes}")
    
    print("✅ Estructura validada: Todas las columnas requeridas están presentes.")
    return True

# --- Uso en tu flujo ---

columnas_obligatorias = [
    'orden_id', 'cliente_id', 'fecha_pedido', 'fecha_entrega', 
    'fecha_proceso', 'canal_venta', 'estado', 'monto_total'
]

# Ejecutamos la validación inmediatamente después de leer el CSV
try:
    verificar_columnas_requeridas(df_detordenes, columnas_obligatorias)
except KeyError as e:
    print(e)
    
    
def validar_longitud_fija(df, columna, nombre_regla):
    """
    Valida que todos los valores en la columna tengan la misma longitud.
    """
    # Convertimos a string y calculamos longitudes
    longitudes = df[columna].astype(str).str.len()
    
    min_len = longitudes.min()
    max_len = longitudes.max()
    
    if min_len != max_len:
        print(f"❌ Error en '{columna}': La longitud no es fija. Mín: {min_len}, Máx: {max_len}")
        
        # Identificamos las filas que no tienen la longitud "dominante" (la más común)
        longitud_correcta = longitudes.mode()[0]
        registros_erroneos = df[longitudes != longitud_correcta]
        
        print(f"   Se encontraron {len(registros_erroneos)} registros con longitud anómala.")
        return False, registros_erroneos
    else:
        print(f"✅ Validación de '{columna}' exitosa: Todos los registros tienen longitud {min_len}.")
        return True, None

# --- Aplicación a tus columnas ---

# Validamos orden_id
es_valido_orden, errores_orden = validar_longitud_fija(df_detordenes, 'orden_id', 'Longitud Orden ID')

# Validamos cliente_id
es_valido_cliente, errores_cliente = validar_longitud_fija(df_detordenes, 'cliente_id', 'Longitud Cliente ID')


import pandas as pd

def procesar_fechas_multiples(df, lista_columnas):
    """
    Convierte múltiples columnas a datetime, preservando originales 
    y reportando errores de formato.
    """
    reporte_calidad = {}
    
    for col in lista_columnas:
        if col in df.columns:
            # 1. Respaldo de seguridad para auditoría
            df[f'{col}_original'] = df[col]
            
            # 2. Limpieza inteligente:
            # Intento 1: Formato estándar YYYY-MM-DD
            fechas = pd.to_datetime(df[col], errors='coerce', format='%Y-%m-%d')
            
            # Intento 2: Formato europeo DD/MM/YYYY solo en los fallidos
            mask_nat = fechas.isna()
            if mask_nat.any():
                fechas.loc[mask_nat] = pd.to_datetime(
                    df.loc[mask_nat, col], 
                    errors='coerce', 
                    format='%d/%m/%Y'
                )
            
            # Asignar la columna procesada
            df[col] = fechas
            
            # 3. Registrar cuántos fallaron para el JSON de reporte
            errores = int(df[col].isna().sum())
            reporte_calidad[col] = errores
            
    return df, reporte_calidad

# --- Cómo usarla en tu flujo ---
columnas_fechas = ['fecha_pedido', 'fecha_entrega', 'fecha_proceso']

# Ejecución
df_detordenes, reporte_fechas = procesar_fechas_multiples(df_detordenes, columnas_fechas)

print("✅ Fechas procesadas. Resumen de errores por columna:")
print(reporte_fechas)



def estandarizar_estados(df, columna, estados_validos):
    # 1. Limpieza básica: quitar espacios y convertir a mayúsculas
    df[columna] = df[columna].str.strip().str.upper()
    
    # 2. Reemplazo: Si el valor no está en la lista de estados_validos, lo cambia a "OTROS"
    # Usamos .apply() para una lógica personalizada
    df[columna] = df[columna].apply(lambda x: x if x in estados_validos else "OTROS")
    
    return df

# --- Uso en tu flujo ---

# Definimos los estados que sí son válidos para tu negocio
lista_estados_permitidos = ['ENTREGADO', 'ANULADO', 'PENDIENTE', 'DEVUELTO']

# Aplicamos la función
df_detordenes = estandarizar_estados(df_detordenes, 'estado', lista_estados_permitidos)

# Verificamos
print(df_detordenes['estado'].value_counts())


def validar_montos(df):
    """
    Valida que todos los montos sean mayores a 0.
    Retorna el df filtrado y un reporte de errores.
    """
    # Identificar inválidos
    invalidos = df[df['monto_total'] <= 0].copy()
    
    # Filtrar válidos
    df_limpio = df[df['monto_total'] > 0].copy()
    
    # Registrar reporte
    reporte_error = {
        "cantidad_invalidos": len(invalidos),
        "promedio_monto_invalido": float(invalidos['monto_total'].mean()) if not invalidos.empty else 0
    }
    
    return df_limpio, invalidos, reporte_error

# --- Integración ---
df_limpio, df_sucios_monto, info_montos = validar_montos(df_detordenes)

if info_montos["cantidad_invalidos"] > 0:
    print(f"⚠️ Alerta: Se detectaron {info_montos['cantidad_invalidos']} registros con monto <= 0.")

Cantidad de filas 5787 y columnas 8
📊 Resumen de validación de esquemas:
✅ Columnas correctas: ['orden_id', 'cliente_id', 'estado', 'monto_total']
❌ Columnas con tipo incorrecto: ['fecha_pedido', 'fecha_entrega', 'fecha_proceso']
⚠️ Acción requerida: Debe convertir las columnas listadas en rojo antes de proceder.
✅ Estructura validada: Todas las columnas requeridas están presentes.
✅ Validación de 'orden_id' exitosa: Todos los registros tienen longitud 10.
✅ Validación de 'cliente_id' exitosa: Todos los registros tienen longitud 6.
✅ Fechas procesadas. Resumen de errores por columna:
{'fecha_pedido': 0, 'fecha_entrega': 0, 'fecha_proceso': 0}
estado
ENTREGADO    5094
ANULADO       284
PENDIENTE     238
DEVUELTO      171
Name: count, dtype: int64
⚠️ Alerta: Se detectaron 38 registros con monto <= 0.


# **Clientes**

In [3]:
url_clientes=r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\1.raw\clientes.csv'
df_cliente=pd.read_csv(url_clientes,sep=',')
print(f'Cantidad de filas {df_cliente.shape[0]} y columnas {df_cliente.shape[1]}')

Cantidad de filas 820 y columnas 7


In [31]:
url_clientes=r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\1.raw\clientes.csv'
df_cliente=pd.read_csv(url_clientes,sep=',')
print(f'Cantidad de filas {df_cliente.shape[0]} y columnas {df_cliente.shape[1]}')
print(df_cliente.isnull().sum()) # identificar los nulls
def validar_columnas_existentes(df):
    """
    Validar si las columnas obligatorias existen en el DataFrame.
    """
    # Columnas que obligatoriamente deben estar
    columnas_requeridas = [
        'cliente_id', 'nombre_comercial', 'canal', 
        'region', 'distrito', 'fecha_alta', 'estado'
    ]
    
    # Obtenemos cuáles de las requeridas NO están en el DataFrame
    columnas_faltantes = [col for col in columnas_requeridas if col not in df.columns]
    
    if not columnas_faltantes:
        print("✅ ¡Todo correcto! Todas las columnas requeridas están presentes en el DataFrame.")
        return True
    else:
        print("❌ ¡Error! Faltan columnas obligatorias en la tabla.")
        print(f"   - Columnas que faltan: {columnas_faltantes}")
        return False
    
validar_columnas_existentes(df_cliente)


def validar_tipos_datos(df):
    """
    Valida que:
    - cliente_id, nombre_comercial, canal, region, distrito, estado sean 'object' o 'str'.
    - fecha_alta sea de tipo datetime.
    """
    # Definimos lo que esperamos
    columnas_texto = ['cliente_id', 'nombre_comercial', 'canal', 'region', 'distrito', 'estado']
    columna_fecha = 'fecha_alta'
    
    errores = []
    
    # 1. Validar columnas de texto (object o str)
    for col in columnas_texto:
        if col in df.columns:
            tipo_actual = str(df[col].dtype)
            # Verificamos si es object o contiene 'string'/'str'
            if not ('object' in tipo_actual or 'string' in tipo_actual or 'str' in tipo_actual):
                errores.append(f"Columna '{col}': se esperaba 'str' u 'object' pero es '{tipo_actual}'")
        else:
            errores.append(f"Columna '{col}': NO existe en el DataFrame.")
            
    # 2. Validar columna de fecha
    if columna_fecha in df.columns:
        tipo_actual = str(df[columna_fecha].dtype)
        if not 'datetime' in tipo_actual:
            errores.append(f"Columna '{columna_fecha}': se esperaba 'datetime' pero es '{tipo_actual}'")
    else:
        errores.append(f"Columna '{columna_fecha}': NO existe en el DataFrame.")
        
    # --- RESULTADO DE LA VALIDACIÓN ---
    if not errores:
        print("✅ ¡Todo correcto! Todos los tipos de datos (str/object y datetime) corresponden al formato.")
        return True
    else:
        print("❌ ¡Error en el formato de tipos de datos!")
        for err in errores:
            print(f"   - {err}")
        return False
    
validar_tipos_datos(df_cliente)

def validar_ids_unicos(df, columna='cliente_id'):
    """
    Valida si todos los valores de la columna de IDs son únicos.
    Imprime un mensaje de éxito o una advertencia si hay duplicados.
    """
    total_filas = len(df)
    valores_unicos = df[columna].nunique()
    
    if valores_unicos == total_filas:
        print(f"✅ ¡Todo correcto! Todos los {valores_unicos} IDs en '{columna}' son únicos.")
        return True
    else:
        duplicados_estimados = total_filas - valores_unicos
        print(f"❌ ¡Alerta! Hay IDs duplicados en la columna '{columna}'.")
        print(f"   - Total de filas: {total_filas}")
        print(f"   - IDs únicos: {valores_unicos}")
        print(f"   - Se estiman al menos {duplicados_estimados} filas repetidas.")
        return False

validar_ids_unicos(df_cliente)   

df_cliente['fecha_alta'] = pd.to_datetime(df_cliente['fecha_alta'], format='%Y-%m-%d', errors='coerce')

def validar_y_limpiar_estados(df, columna='estado'):
    """
    Valida la columna de estados. 
    - Si solo hay ACTIVO e INACTIVO, imprime un mensaje de éxito.
    - Si hay otros estados, los limpia a 'OTROS' y avisa cuántos cambió.
    """
    df_copia = df.copy()
    estados_validos = {'ACTIVO', 'INACTIVO'}
    
    # Obtenemos los estados únicos que realmente existen en el DataFrame
    estados_actuales = set(df_copia[columna].dropna().unique())
    
    # Verificamos si hay estados que NO pertenecen a los válidos
    estados_incorrectos = estados_actuales - estados_validos
    
    if not estados_incorrectos:
        print("✅ ¡Todo correcto! La columna solo contiene ACTIVO e INACTIVO.")
    else:
        print(f"⚠️ Se detectaron estados incorrectos: {estados_incorrectos}")
        print("🔄 Limpiando y reemplazando por 'OTROS'...")
        
        # Aplicamos el reemplazo solo si hubo errores
        df_copia[columna] = df_copia[columna].where(df_copia[columna].isin(estados_validos), 'OTROS')
    
    return df_copia

validar_estado=validar_y_limpiar_estados(df_cliente)
validar_estado


def transformar_columnas_a_minusculas(df):
    """
    Convierte a minúsculas las columnas 'region', 'distrito' y 'canal'.
    Muestra los value_counts() para confirmar el cambio.
    """
    df_copia = df.copy()
    columnas_a_transformar = ['region', 'distrito', 'canal']
    
    print("🔄 Transformando columnas a minúsculas...")
    
    for col in columnas_a_transformar:
        if col in df_copia.columns:
            # .str.lower() convierte todo el texto de la columna a minúsculas
            df_copia[col] = df_copia[col].str.lower()
            
            # Mostramos el resultado del conteo
            print(f"\n📊 Conteo de valores para '{col}':")
            print(df_copia[col].value_counts())
            print("-" * 40)
        else:
            print(f"⚠️ La columna '{col}' no se encontró en el DataFrame.")
            
    print("✅ ¡Transformación completada con éxito!")
    return df_copia

transformar_columnas_a_minusculas(df_cliente)







Cantidad de filas 820 y columnas 7
cliente_id           0
nombre_comercial     0
canal                0
region               0
distrito             0
fecha_alta          12
estado               0
dtype: int64
✅ ¡Todo correcto! Todas las columnas requeridas están presentes en el DataFrame.
❌ ¡Error en el formato de tipos de datos!
   - Columna 'fecha_alta': se esperaba 'datetime' pero es 'str'
✅ ¡Todo correcto! Todos los 820 IDs en 'cliente_id' son únicos.
✅ ¡Todo correcto! La columna solo contiene ACTIVO e INACTIVO.
🔄 Transformando columnas a minúsculas...

📊 Conteo de valores para 'region':
region
lima sur        95
piura           93
callao          92
cusco           91
arequipa        90
lima centro     86
lima norte      84
trujillo        82
lima este       82
lima norte       4
cusco            4
piura            4
lima centro      3
lima este        3
lima-n           2
arequipa         2
trujillo         1
lima-c           1
lima-s           1
Name: count, dtype: int64
-------

,cliente_id,nombre_comercial,canal,region,distrito,fecha_alta,estado
0,C00001,Bodega Ramos 115,bodega,callao,cercado,2018-04-13,ACTIVO
1,C00002,Restaurante Ramos 759,restaurante/bar,lima sur,comas,2024-02-12,ACTIVO
2,C00003,Minimarket García 239,minimarket,lima norte,los olivos,2023-09-01,ACTIVO
3,C00004,Bodega Castillo 430,bodega,piura,villa el salvador,2020-06-21,INACTIVO
4,C00005,Bodega Díaz 826,bodega,trujillo,cercado,2019-10-16,ACTIVO
...,...,...,...,...,...,...,...
815,C00816,Bodega Torres 766,bodega,cusco,cercado,2024-08-01,ACTIVO
816,C00817,Bodega Quispe 129,bodega,lima este,los olivos,2026-02-28,ACTIVO
817,C00818,Bodega Flores 216,bodega,trujillo,san juan,2019-11-15,INACTIVO
818,C00819,Bodega Sánchez 562,bodega,lima norte,wanchaq,2018-08-09,ACTIVO


# **Producto**

In [45]:
url_producto=r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\1.raw\productos.csv'
df_producto=pd.read_csv(url_producto,sep=',')
print(f'Cantidad de filas {df_producto.shape[0]} y columnas {df_producto.shape[1]}')
print(df_producto.isnull().sum()) 


def validar_tipos_producto(df):
    """
    Valida los tipos de datos de la tabla df_producto:
    - Texto (str/object): producto_id, nombre_producto, categoria, marca, presentacion
    - Decimal (float): contenido_ml, precio_lista
    - Entero (int64): unidades_por_caja
    """
    # 1. Definimos los grupos de columnas esperados
    columnas_texto = ['producto_id', 'nombre_producto', 'categoria', 'marca', 'presentacion']
    columnas_float = ['contenido_ml', 'precio_lista']
    columna_int = 'unidades_por_caja'
    
    errores = []
    
    # 2. Validar columnas de texto
    for col in columnas_texto:
        if col in df.columns:
            tipo_actual = str(df[col].dtype)
            if not ('object' in tipo_actual or 'string' in tipo_actual or 'str' in tipo_actual):
                errores.append(f"Columna '{col}': se esperaba 'str' u 'object' pero es '{tipo_actual}'")
        else:
            errores.append(f"Columna '{col}': NO existe en la tabla.")
            
    # 3. Validar columnas float
    for col in columnas_float:
        if col in df.columns:
            tipo_actual = str(df[col].dtype)
            if not 'float' in tipo_actual:
                errores.append(f"Columna '{col}': se esperaba 'float' pero es '{tipo_actual}'")
        else:
            errores.append(f"Columna '{col}': NO existe en la tabla.")
            
    # 4. Validar columna int64
    if columna_int in df.columns:
        tipo_actual = str(df[columna_int].dtype)
        if not 'int64' in tipo_actual:
            errores.append(f"Columna '{columna_int}': se esperaba 'int64' pero es '{tipo_actual}'")
    else:
        errores.append(f"Columna '{columna_int}': NO existe en la tabla.")
        
    # --- IMPRIMIR RESULTADOS ---
    if not errores:
        print("✅ ¡Todo correcto! Todos los tipos de datos de df_producto coinciden con el formato requerido.")
        return True
    else:
        print("❌ ¡Error en el formato de df_producto!")
        for err in errores:
            print(f"   - {err}")
        return False
    
validar_tipos_producto(df_producto)

Cantidad de filas 121 y columnas 8
producto_id          0
nombre_producto      0
categoria            0
marca                0
presentacion         0
contenido_ml         4
unidades_por_caja    0
precio_lista         0
dtype: int64
✅ ¡Todo correcto! Todos los tipos de datos de df_producto coinciden con el formato requerido.


True

In [57]:
url_producto=r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\1.raw\productos.csv'
df_producto=pd.read_csv(url_producto,sep=',')
print(f'Cantidad de filas {df_producto.shape[0]} y columnas {df_producto.shape[1]}')
print(df_producto.isnull().sum()) 


def validar_tipos_producto(df):
    """
    Valida los tipos de datos de la tabla df_producto:
    - Texto (str/object): producto_id, nombre_producto, categoria, marca, presentacion
    - Decimal (float): contenido_ml, precio_lista
    - Entero (int64): unidades_por_caja
    """
    # 1. Definimos los grupos de columnas esperados
    columnas_texto = ['producto_id', 'nombre_producto', 'categoria', 'marca', 'presentacion']
    columnas_float = ['contenido_ml', 'precio_lista']
    columna_int = 'unidades_por_caja'
    
    errores = []
    
    # 2. Validar columnas de texto
    for col in columnas_texto:
        if col in df.columns:
            tipo_actual = str(df[col].dtype)
            if not ('object' in tipo_actual or 'string' in tipo_actual or 'str' in tipo_actual):
                errores.append(f"Columna '{col}': se esperaba 'str' u 'object' pero es '{tipo_actual}'")
        else:
            errores.append(f"Columna '{col}': NO existe en la tabla.")
            
    # 3. Validar columnas float
    for col in columnas_float:
        if col in df.columns:
            tipo_actual = str(df[col].dtype)
            if not 'float' in tipo_actual:
                errores.append(f"Columna '{col}': se esperaba 'float' pero es '{tipo_actual}'")
        else:
            errores.append(f"Columna '{col}': NO existe en la tabla.")
            
    # 4. Validar columna int64
    if columna_int in df.columns:
        tipo_actual = str(df[columna_int].dtype)
        if not 'int64' in tipo_actual:
            errores.append(f"Columna '{columna_int}': se esperaba 'int64' pero es '{tipo_actual}'")
    else:
        errores.append(f"Columna '{columna_int}': NO existe en la tabla.")
        
    # --- IMPRIMIR RESULTADOS ---
    if not errores:
        print("✅ ¡Todo correcto! Todos los tipos de datos de df_producto coinciden con el formato requerido.")
        return True
    else:
        print("❌ ¡Error en el formato de df_producto!")
        for err in errores:
            print(f"   - {err}")
        return False
    
validar_tipos_producto(df_producto)


def validar_productos_unicos(df, columna='producto_id'):
    """
    Valida si todos los valores de producto_id en la tabla df_producto son únicos.
    Imprime un mensaje de éxito o una alerta con los duplicados encontrados.
    """
    total_filas = len(df)
    valores_unicos = df[columna].nunique()
    
    if valores_unicos == total_filas:
        print(f"✅ ¡Todo correcto! Los {valores_unicos} productos en '{columna}' son únicos.")
        return True
    else:
        duplicados_estimados = total_filas - valores_unicos
        print(f"❌ ¡Alerta! Se encontraron productos duplicados en la columna '{columna}'.")
        print(f"   - Total de filas en la tabla: {total_filas}")
        print(f"   - IDs de producto únicos: {valores_unicos}")
        print(f"   - Hay al menos {duplicados_estimados} registros repetidos.")
        
        # Opcional: Mostrar cuáles son los IDs que se están repitiendo
        ids_repetidos = df[df.duplicated(subset=[columna], keep=False)][columna].unique()
        print(f"   - Algunos de los IDs duplicados son: {list(ids_repetidos[:5])}") # Muestra los primeros 5
        
        return False
    
validar_productos_unicos(df_producto)


def validar_columnas_producto(df):
    """
    Valida si las columnas obligatorias existen en la tabla df_producto.
    """
    # Lista de columnas requeridas para la tabla de productos
    columnas_requeridas = [
        'producto_id', 'nombre_producto', 'categoria', 'marca', 
        'presentacion', 'contenido_ml', 'unidades_por_caja', 'precio_lista'
    ]
    
    # Identificamos si falta alguna columna
    columnas_faltantes = [col for col in columnas_requeridas if col not in df.columns]
    
    if not columnas_faltantes:
        print("✅ ¡Todo correcto! Todas las columnas obligatorias de la tabla de productos están presentes.")
        return True
    else:
        print("❌ ¡Error! Faltan columnas obligatorias en la tabla df_producto.")
        print(f"   - Columnas que hacen falta: {columnas_faltantes}")
        return False
    
    
validar_columnas_producto(df_producto)

# Buscamos números seguidos opcionalmente de un espacio y las letras ml, l, mL o L
df_producto['volumen'] = df_producto['presentacion'].str.extract(r'(\d+\.?\d*\s*(?:ml|l|L|mL))', expand=False)
import re
def convertir_a_ml(valor):
    # Si el valor es nulo (NaN), lo dejamos como nulo
    if pd.isna(valor):
        return None
    
    # Convertimos a string y quitamos espacios
    valor_str = str(valor).strip().lower()
    
    # Extraemos solo el número (entero o decimal)
    numero = float(re.search(r'\d+\.?\d*', valor_str).group())
    
    # Si contiene 'ml', nos quedamos con el número exacto
    if 'ml' in valor_str:
        return numero
    # Si contiene solo 'l' (pero no 'ml'), multiplicamos por 1000
    elif 'l' in valor_str:
        return numero * 1000
    
    return numero

df_producto['contenido_ml_new'] = df_producto['volumen'].apply(convertir_a_ml)
df_producto



Cantidad de filas 121 y columnas 8
producto_id          0
nombre_producto      0
categoria            0
marca                0
presentacion         0
contenido_ml         4
unidades_por_caja    0
precio_lista         0
dtype: int64
✅ ¡Todo correcto! Todos los tipos de datos de df_producto coinciden con el formato requerido.
❌ ¡Alerta! Se encontraron productos duplicados en la columna 'producto_id'.
   - Total de filas en la tabla: 121
   - IDs de producto únicos: 120
   - Hay al menos 1 registros repetidos.
   - Algunos de los IDs duplicados son: ['P-0042']
✅ ¡Todo correcto! Todas las columnas obligatorias de la tabla de productos están presentes.


,producto_id,nombre_producto,categoria,marca,presentacion,contenido_ml,unidades_por_caja,precio_lista,volumen,contenido_ml_new
0,P-0001,Misti Cola 355ml Lata,Gaseosa Cola,Misti Cola,355ml Lata,355.0,12,7.52,355ml,355.0
1,P-0002,Inti Cola 1L Botella,Gaseosa Cola,Inti Cola,1L Botella,1000.0,24,8.32,1L,1000.0
2,P-0003,Inti Cola 3L Botella,Gaseosa Cola,Inti Cola,3L Botella,3000.0,24,4.61,3L,3000.0
3,P-0004,Kola Andina 3L Botella,Gaseosa Cola,Kola Andina,3L Botella,3000.0,24,5.68,3L,3000.0
4,P-0005,Inti Cola 3L Botella,Gaseosa Cola,Inti Cola,3L Botella,3000.0,24,5.98,3L,3000.0
...,...,...,...,...,...,...,...,...,...,...
116,P-0117,Hidra+ 500ml Botella,Isotónica,Hidra+,500ml Botella,0.5,24,4.38,500ml,500.0
117,P-0118,Andes Sport 1.5L Botella,Isotónica,Andes Sport,1.5L Botella,1.5,30,4.65,1.5L,1500.0
118,P-0119,Hidra+ 3L Botella,Isotónica,Hidra+,3L Botella,3.0,24,4.43,3L,3000.0
119,P-0120,Andes Sport 500ml Botella,Isotónica,Andes Sport,500ml Botella,500.0,24,4.83,500ml,500.0


In [56]:
# Buscamos números seguidos opcionalmente de un espacio y las letras ml, l, mL o L
df_producto['volumen'] = df_producto['presentacion'].str.extract(r'(\d+\.?\d*\s*(?:ml|l|L|mL))', expand=False)
import re
def convertir_a_ml(valor):
    # Si el valor es nulo (NaN), lo dejamos como nulo
    if pd.isna(valor):
        return None
    
    # Convertimos a string y quitamos espacios
    valor_str = str(valor).strip().lower()
    
    # Extraemos solo el número (entero o decimal)
    numero = float(re.search(r'\d+\.?\d*', valor_str).group())
    
    # Si contiene 'ml', nos quedamos con el número exacto
    if 'ml' in valor_str:
        return numero
    # Si contiene solo 'l' (pero no 'ml'), multiplicamos por 1000
    elif 'l' in valor_str:
        return numero * 1000
    
    return numero

df_producto['contenido_ml_new'] = df_producto['volumen'].apply(convertir_a_ml)
df_producto

,producto_id,nombre_producto,categoria,marca,presentacion,contenido_ml,unidades_por_caja,precio_lista,volumen,contenido_ml_new
0,P-0001,Misti Cola 355ml Lata,Gaseosa Cola,Misti Cola,355ml Lata,355.0,12,7.52,355ml,355.0
1,P-0002,Inti Cola 1L Botella,Gaseosa Cola,Inti Cola,1L Botella,1000.0,24,8.32,1L,1000.0
2,P-0003,Inti Cola 3L Botella,Gaseosa Cola,Inti Cola,3L Botella,3000.0,24,4.61,3L,3000.0
3,P-0004,Kola Andina 3L Botella,Gaseosa Cola,Kola Andina,3L Botella,3000.0,24,5.68,3L,3000.0
4,P-0005,Inti Cola 3L Botella,Gaseosa Cola,Inti Cola,3L Botella,3000.0,24,5.98,3L,3000.0
...,...,...,...,...,...,...,...,...,...,...
116,P-0117,Hidra+ 500ml Botella,Isotónica,Hidra+,500ml Botella,0.5,24,4.38,500ml,500.0
117,P-0118,Andes Sport 1.5L Botella,Isotónica,Andes Sport,1.5L Botella,1.5,30,4.65,1.5L,1500.0
118,P-0119,Hidra+ 3L Botella,Isotónica,Hidra+,3L Botella,3.0,24,4.43,3L,3000.0
119,P-0120,Andes Sport 500ml Botella,Isotónica,Andes Sport,500ml Botella,500.0,24,4.83,500ml,500.0


# **Carga de datos**

# **load cliente**

In [69]:
import os
import pandas as pd
import pyodbc
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd
import os
import json

def procesar_cliente(ruta_archivo):
    # 1. Carga
    df = pd.read_csv(ruta_archivo, sep=',')
    
    # 1. Quitar los espacios en blanco (izquierda y derecha)
    df['region'] = df['region'].str.strip()

    # 2. Definir el diccionario con los cambios requeridos
    cambios_region = {
        'lima-c': 'lima centro',
        'lima-n': 'lima norte',
        'lima-s': 'lima sur'
    }

    # 3. Reemplazar los valores en la columna
    df['region'] = df['region'].replace(cambios_region)
    
    cols_necesarias = ['cliente_id', 'nombre_comercial', 'canal', 'region', 'distrito', 'fecha_alta','estado']
    df = df[cols_necesarias].copy()
    
    # 2. Renombrar
    df = df.rename(columns={'fecha_alta_original': 'fecha_alta', 'estado_original': 'estado'})
    
    # 3. Procesamiento de fechas 
    df['fecha_alta'] = pd.to_datetime(df['fecha_alta'], errors='coerce', format='%Y-%m-%d')
    
    # --- NUEVA LÓGICA DE METADATOS EN JSON ---
    nombre_archivo = os.path.basename(ruta_archivo) # Extrae 'clientes_cleaned.csv' de la ruta completa
    numero_filas = len(df)                          # Cuenta el total de filas
    fecha_carga_str = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S') # Fecha actual en texto para el JSON
    
    # Creamos el diccionario que se convertirá en JSON
    diccionario_metadatos = {
        "nombre_archivo": nombre_archivo,
        "numero_filas": numero_filas,
        "fecha_carga": fecha_carga_str
    }
    
    # json.dumps convierte el diccionario de Python a un String JSON válido
    df['metadatos_json'] = json.dumps(diccionario_metadatos)
    # -----------------------------------------
    
    # Columnas de auditoría originales
    df['source_file'] = nombre_archivo        # 'archivo_origen' -> 'source_file'
    df['load_date'] = pd.Timestamp.now()      # 'fecha_carga' -> 'load_date'
    df['update_date'] = pd.Timestamp.now()    # 'fecha_act' -> 'update_date'
    df['loaded_by'] = 'proceso_etl_python'    # 'usuario_carga' -> 'loaded_by'
    df['is_active'] = 1                       # 'activo' -> 'is_active'
    return df

# --- Ejecución ---
url = r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\2.processed\clientes_cleaned.csv'
df_cliente = procesar_cliente(url)
df_cliente

,cliente_id,nombre_comercial,canal,region,distrito,fecha_alta,estado,metadatos_json,source_file,load_date,update_date,loaded_by,is_active
0,C00001,Bodega Ramos 115,bodega,callao,cercado,2018-04-13,ACTIVO,"{""nombre_archivo"": ""clientes_cleaned.csv"", ""nu...",clientes_cleaned.csv,2026-06-22 00:41:54.425535,2026-06-22 00:41:54.425961,proceso_etl_python,1
1,C00002,Restaurante Ramos 759,restaurante/bar,lima sur,comas,2024-02-12,ACTIVO,"{""nombre_archivo"": ""clientes_cleaned.csv"", ""nu...",clientes_cleaned.csv,2026-06-22 00:41:54.425535,2026-06-22 00:41:54.425961,proceso_etl_python,1
2,C00003,Minimarket García 239,minimarket,lima norte,los olivos,2023-09-01,ACTIVO,"{""nombre_archivo"": ""clientes_cleaned.csv"", ""nu...",clientes_cleaned.csv,2026-06-22 00:41:54.425535,2026-06-22 00:41:54.425961,proceso_etl_python,1
3,C00004,Bodega Castillo 430,bodega,piura,villa el salvador,2020-06-21,INACTIVO,"{""nombre_archivo"": ""clientes_cleaned.csv"", ""nu...",clientes_cleaned.csv,2026-06-22 00:41:54.425535,2026-06-22 00:41:54.425961,proceso_etl_python,1
4,C00005,Bodega Díaz 826,bodega,trujillo,cercado,2019-10-16,ACTIVO,"{""nombre_archivo"": ""clientes_cleaned.csv"", ""nu...",clientes_cleaned.csv,2026-06-22 00:41:54.425535,2026-06-22 00:41:54.425961,proceso_etl_python,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
815,C00816,Bodega Torres 766,bodega,cusco,cercado,2024-08-01,ACTIVO,"{""nombre_archivo"": ""clientes_cleaned.csv"", ""nu...",clientes_cleaned.csv,2026-06-22 00:41:54.425535,2026-06-22 00:41:54.425961,proceso_etl_python,1
816,C00817,Bodega Quispe 129,bodega,lima este,los olivos,2026-02-28,ACTIVO,"{""nombre_archivo"": ""clientes_cleaned.csv"", ""nu...",clientes_cleaned.csv,2026-06-22 00:41:54.425535,2026-06-22 00:41:54.425961,proceso_etl_python,1
817,C00818,Bodega Flores 216,bodega,trujillo,san juan,2019-11-15,INACTIVO,"{""nombre_archivo"": ""clientes_cleaned.csv"", ""nu...",clientes_cleaned.csv,2026-06-22 00:41:54.425535,2026-06-22 00:41:54.425961,proceso_etl_python,1
818,C00819,Bodega Sánchez 562,bodega,lima norte,wanchaq,2018-08-09,ACTIVO,"{""nombre_archivo"": ""clientes_cleaned.csv"", ""nu...",clientes_cleaned.csv,2026-06-22 00:41:54.425535,2026-06-22 00:41:54.425961,proceso_etl_python,1


In [70]:
df_cliente['region'].value_counts()

region
piura          97
lima sur       96
cusco          95
callao         92
arequipa       92
lima norte     90
lima centro    90
lima este      85
trujillo       83
Name: count, dtype: int64

In [19]:
import os
import pandas as pd
import pyodbc
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd
import os
import json

def procesar_cliente(ruta_archivo):
    # 1. Carga
    df = pd.read_csv(ruta_archivo, sep=',')
    
    cols_necesarias = ['cliente_id', 'nombre_comercial', 'canal', 'region', 'distrito', 'fecha_alta','estado']
    df = df[cols_necesarias].copy()
    
    # 2. Renombrar
    df = df.rename(columns={'fecha_alta_original': 'fecha_alta', 'estado_original': 'estado'})
    
    # 3. Procesamiento de fechas 
    df['fecha_alta'] = pd.to_datetime(df['fecha_alta'], errors='coerce', format='%Y-%m-%d')
    
    # --- NUEVA LÓGICA DE METADATOS EN JSON ---
    nombre_archivo = os.path.basename(ruta_archivo) # Extrae 'clientes_cleaned.csv' de la ruta completa
    numero_filas = len(df)                          # Cuenta el total de filas
    fecha_carga_str = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S') # Fecha actual en texto para el JSON
    
    # Creamos el diccionario que se convertirá en JSON
    diccionario_metadatos = {
        "nombre_archivo": nombre_archivo,
        "numero_filas": numero_filas,
        "fecha_carga": fecha_carga_str
    }
    
    # json.dumps convierte el diccionario de Python a un String JSON válido
    df['metadatos_json'] = json.dumps(diccionario_metadatos)
    # -----------------------------------------
    
    # Columnas de auditoría originales
    df['source_file'] = nombre_archivo        # 'archivo_origen' -> 'source_file'
    df['load_date'] = pd.Timestamp.now()      # 'fecha_carga' -> 'load_date'
    df['update_date'] = pd.Timestamp.now()    # 'fecha_act' -> 'update_date'
    df['loaded_by'] = 'proceso_etl_python'    # 'usuario_carga' -> 'loaded_by'
    df['is_active'] = 1                       # 'activo' -> 'is_active'
    return df

# --- Ejecución ---
url = r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\2.processed\clientes_cleaned.csv'
df_cliente = procesar_cliente(url)
df_cliente

# 1. Cargar variables de entorno del archivo .env
load_dotenv()
server = os.getenv('DB_SERVER')
database = os.getenv('DB_DATABASE')

# 2. Validar que las variables existan
if not server or not database:
    raise ValueError("❌ No se encontraron las variables de entorno DB_SERVER o DB_DATABASE.")

print(f"🔄 Conectando al servidor '{server}' en la base de datos '{database}'...")

# 3. Crear la cadena de conexión para SQL Server (Autenticación de Windows)
# Usamos el Driver 17, que es el estándar más compatible
connection_string = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"

# Crear el motor de conexión (Engine) de SQLAlchemy
engine = create_engine(connection_string)

try:
    # --- Supongamos que aquí ya tienes tu 'df_cliente' procesado de los pasos anteriores ---
    # (Descomenta la siguiente línea en tu código real si vas a usar el tuyo)
    # df_cliente = procesar_cliente('tu_archivo.csv') 
    
    print(f"🚀 Iniciando la carga de {len(df_cliente)} filas a la tabla 'cliente'...")
    
    # 4. Subir a SQL Server en bloques
    df_cliente.to_sql(
        name='cliente',            # Nombre exacto de la tabla en SQL Server
        con=engine,                # Nuestro motor de conexión
        if_exists='append',        # 'append' agrega los datos (no borra la tabla existente)
        index=False,               # No sube el índice de Pandas como una columna
        chunksize=1000,            # Manda los datos en bloques de 1,000 en 1,000 filas
    )
    
    print("✅ ¡Carga completada con éxito en bloques de 1000!")

except Exception as e:
    print(f"❌ Ocurrió un error al subir los datos: {e}")

🔄 Conectando al servidor 'DESKTOP-BLERSIO' en la base de datos 'backus'...
🚀 Iniciando la carga de 819 filas a la tabla 'cliente'...
✅ ¡Carga completada con éxito en bloques de 1000!


# **load producto**

In [11]:
import os
import pandas as pd
import pyodbc
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd
import os
import json


def procesar_producto(ruta_archivo):
    # 1. Carga del archivo de productos
    df_producto = pd.read_csv(ruta_archivo, sep=',')
    
    # 2. Selección de columnas necesarias
    cols_necesarias = [
        'producto_id', 'nombre_producto', 'categoria', 'marca', 'presentacion',
        'volumen', 'contenido_ml_new', 'unidades_por_caja', 'precio_lista'
    ]
    df_producto = df_producto[cols_necesarias].copy()
    
    # 3. Renombrar columnas
    df_producto = df_producto.rename(columns={'contenido_ml_new': 'contenido_ml'})
    
    # --- LÓGICA DE METADATOS EN JSON ---
    nombre_archivo = os.path.basename(ruta_archivo)  # Extrae el nombre del archivo de la ruta
    numero_filas = len(df_producto)                  # Cuenta el total de filas del DataFrame de productos
    fecha_carga_str = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    
    diccionario_metadatos = {
        "nombre_archivo": nombre_archivo,
        "numero_filas": numero_filas,
        "fecha_carga": fecha_carga_str
    }
    
    # Convierte el diccionario a un String JSON válido
    df_producto['metadatos_json'] = json.dumps(diccionario_metadatos)
    # -------------------------------------
    
    # 4. Columnas de auditoría estandarizadas en inglés (coincidentes con tu modelo)
    df_producto['source_file'] = nombre_archivo
    df_producto['load_date'] = pd.Timestamp.now()
    df_producto['update_date'] = pd.Timestamp.now()
    df_producto['loaded_by'] = 'proceso_etl_python'
    df_producto['is_active'] = 1
    
    return df_producto

# --- Ejecución de la función ---
url_producto = r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\2.processed\productos_cleaned.csv'
df_producto = procesar_producto(url_producto)
# 1. Cargar variables de entorno del archivo .env
load_dotenv()
server = os.getenv('DB_SERVER')
database = os.getenv('DB_DATABASE')

# 2. Validar que las variables existan
if not server or not database:
    raise ValueError("❌ No se encontraron las variables de entorno DB_SERVER o DB_DATABASE.")

print(f"🔄 Conectando al servidor '{server}' en la base de datos '{database}'...")

# 3. Crear la cadena de conexión para SQL Server (Autenticación de Windows)
# Usamos el Driver 17, que es el estándar más compatible
connection_string = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"

# Crear el motor de conexión (Engine) de SQLAlchemy
engine = create_engine(connection_string)

try:    
    print(f"🚀 Iniciando la carga de {len(df_producto)} filas a la tabla 'cliente'...")
    
    # 4. Subir a SQL Server en bloques
    df_producto.to_sql(
        name='producto',            # Nombre exacto de la tabla en SQL Server
        con=engine,                # Nuestro motor de conexión
        if_exists='append',        # 'append' agrega los datos (no borra la tabla existente)
        index=False,               # No sube el índice de Pandas como una columna
        chunksize=1000,            # Manda los datos en bloques de 1,000 en 1,000 filas
    )
    
    print("✅ ¡Carga completada con éxito en bloques de 1000!")

except Exception as e:
    print(f"❌ Ocurrió un error al subir los datos: {e}")

,producto_id,nombre_producto,categoria,marca,presentacion,volumen,contenido_ml,unidades_por_caja,precio_lista,metadatos_json,source_file,load_date,update_date,loaded_by,is_active
0,P-0001,Misti Cola 355ml Lata,Gaseosa Cola,Misti Cola,355ml Lata,355ml,355.0,12,7.52,"{""nombre_archivo"": ""productos_cleaned.csv"", ""n...",productos_cleaned.csv,2026-06-21 17:02:36.065011,2026-06-21 17:02:36.065285,proceso_etl_python,1
1,P-0002,Inti Cola 1L Botella,Gaseosa Cola,Inti Cola,1L Botella,1L,1000.0,24,8.32,"{""nombre_archivo"": ""productos_cleaned.csv"", ""n...",productos_cleaned.csv,2026-06-21 17:02:36.065011,2026-06-21 17:02:36.065285,proceso_etl_python,1
2,P-0003,Inti Cola 3L Botella,Gaseosa Cola,Inti Cola,3L Botella,3L,3000.0,24,4.61,"{""nombre_archivo"": ""productos_cleaned.csv"", ""n...",productos_cleaned.csv,2026-06-21 17:02:36.065011,2026-06-21 17:02:36.065285,proceso_etl_python,1
3,P-0004,Kola Andina 3L Botella,Gaseosa Cola,Kola Andina,3L Botella,3L,3000.0,24,5.68,"{""nombre_archivo"": ""productos_cleaned.csv"", ""n...",productos_cleaned.csv,2026-06-21 17:02:36.065011,2026-06-21 17:02:36.065285,proceso_etl_python,1
4,P-0005,Inti Cola 3L Botella,Gaseosa Cola,Inti Cola,3L Botella,3L,3000.0,24,5.98,"{""nombre_archivo"": ""productos_cleaned.csv"", ""n...",productos_cleaned.csv,2026-06-21 17:02:36.065011,2026-06-21 17:02:36.065285,proceso_etl_python,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,P-0116,Hidra+ 500ml Botella,Isotónica,Hidra+,500ml Botella,500ml,500.0,6,3.87,"{""nombre_archivo"": ""productos_cleaned.csv"", ""n...",productos_cleaned.csv,2026-06-21 17:02:36.065011,2026-06-21 17:02:36.065285,proceso_etl_python,1
116,P-0117,Hidra+ 500ml Botella,Isotónica,Hidra+,500ml Botella,500ml,500.0,24,4.38,"{""nombre_archivo"": ""productos_cleaned.csv"", ""n...",productos_cleaned.csv,2026-06-21 17:02:36.065011,2026-06-21 17:02:36.065285,proceso_etl_python,1
117,P-0118,Andes Sport 1.5L Botella,Isotónica,Andes Sport,1.5L Botella,1.5L,1500.0,30,4.65,"{""nombre_archivo"": ""productos_cleaned.csv"", ""n...",productos_cleaned.csv,2026-06-21 17:02:36.065011,2026-06-21 17:02:36.065285,proceso_etl_python,1
118,P-0119,Hidra+ 3L Botella,Isotónica,Hidra+,3L Botella,3L,3000.0,24,4.43,"{""nombre_archivo"": ""productos_cleaned.csv"", ""n...",productos_cleaned.csv,2026-06-21 17:02:36.065011,2026-06-21 17:02:36.065285,proceso_etl_python,1


In [13]:
# 1. Cargar variables de entorno del archivo .env
load_dotenv()
server = os.getenv('DB_SERVER')
database = os.getenv('DB_DATABASE')

# 2. Validar que las variables existan
if not server or not database:
    raise ValueError("❌ No se encontraron las variables de entorno DB_SERVER o DB_DATABASE.")

print(f"🔄 Conectando al servidor '{server}' en la base de datos '{database}'...")

# 3. Crear la cadena de conexión para SQL Server (Autenticación de Windows)
# Usamos el Driver 17, que es el estándar más compatible
connection_string = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"

# Crear el motor de conexión (Engine) de SQLAlchemy
engine = create_engine(connection_string)

try:    
    print(f"🚀 Iniciando la carga de {len(df_producto)} filas a la tabla 'cliente'...")
    
    # 4. Subir a SQL Server en bloques
    df_producto.to_sql(
        name='producto',            # Nombre exacto de la tabla en SQL Server
        con=engine,                # Nuestro motor de conexión
        if_exists='append',        # 'append' agrega los datos (no borra la tabla existente)
        index=False,               # No sube el índice de Pandas como una columna
        chunksize=1000,            # Manda los datos en bloques de 1,000 en 1,000 filas
    )
    
    print("✅ ¡Carga completada con éxito en bloques de 1000!")

except Exception as e:
    print(f"❌ Ocurrió un error al subir los datos: {e}")

🔄 Conectando al servidor 'DESKTOP-BLERSIO' en la base de datos 'backus'...
🚀 Iniciando la carga de 120 filas a la tabla 'cliente'...
✅ ¡Carga completada con éxito en bloques de 1000!


# **Load Ordenes**

In [37]:
url_ordenes=r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\2.processed\ordenes_cleaned.csv'
df_ordenes=pd.read_csv(url_ordenes,sep=',')
df_ordenes=df_ordenes[['orden_id', 'cliente_id', 'fecha_pedido', 'fecha_entrega','fecha_proceso', 'canal_venta', 'estado', 'monto_total']]
df_ordenes

,orden_id,cliente_id,fecha_pedido,fecha_entrega,fecha_proceso,canal_venta,estado,monto_total
0,ORD-004432,C00007,2026-02-01,2026-02-04,2026-02-06,DISTRIBUIDOR,ENTREGADO,17.38
1,ORD-003518,C00550,2025-12-04,2025-12-06,2025-12-07,APP B2B,ENTREGADO,394.32
2,ORD-003064,C00040,2025-12-06,2025-12-06,2025-12-06,VENDEDOR,ENTREGADO,438.40
3,ORD-004048,C00097,2026-02-25,2026-02-28,2026-02-28,APP B2B,ENTREGADO,680.92
4,ORD-003039,C00232,2025-12-30,2026-01-02,2026-01-02,VENDEDOR,ENTREGADO,18.44
...,...,...,...,...,...,...,...,...
5597,ORD-002575,C00050,2025-11-13,2025-11-15,2025-11-16,APP B2B,ENTREGADO,48.60
5598,ORD-002551,C00306,2025-11-13,2025-11-13,2025-11-14,VENDEDOR,ENTREGADO,861.49
5599,ORD-000538,C00283,2025-07-01,2025-07-04,2025-07-04,VENDEDOR,ENTREGADO,640.87
5600,ORD-001221,C00606,2025-08-20,2025-08-22,2025-08-23,APP B2B,ENTREGADO,291.67


In [32]:
# 1. Agrupamos por orden_id y contamos los estados únicos
estados_por_orden = df_ordenes.groupby('orden_id')['estado'].nunique().reset_index()

# 2. Renombramos la columna para que sea clara
estados_por_orden = estados_por_orden.rename(columns={'estado': 'cantidad_estados'})

# 3. Filtramos para ver solo las órdenes que tienen 2 o más estados a la vez
ordenes_multi_estado = estados_por_orden[estados_por_orden['cantidad_estados'] > 1]

# Mostrar el resultado
print(f"Cantidad de órdenes con múltiples estados: {len(ordenes_multi_estado)}")
print(ordenes_multi_estado)

Cantidad de órdenes con múltiples estados: 0
Empty DataFrame
Columns: [orden_id, cantidad_estados]
Index: []


In [40]:
df_ordenes.info()
url_ordenes=r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\2.processed\ordenes_cleaned.csv'
df_ordenes=pd.read_csv(url_ordenes,sep=',')
df_ordenes=df_ordenes[['orden_id', 'cliente_id', 'fecha_pedido', 'fecha_entrega','fecha_proceso', 'canal_venta', 'estado', 'monto_total']]


<class 'pandas.DataFrame'>
RangeIndex: 5602 entries, 0 to 5601
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   orden_id       5602 non-null   str    
 1   cliente_id     5602 non-null   str    
 2   fecha_pedido   5602 non-null   str    
 3   fecha_entrega  5602 non-null   str    
 4   fecha_proceso  5602 non-null   str    
 5   canal_venta    5602 non-null   str    
 6   estado         5602 non-null   str    
 7   monto_total    5602 non-null   float64
dtypes: float64(1), str(7)
memory usage: 350.3 KB


In [45]:
def procesar_ordenes(ruta_archivo):
    # 1. Carga del archivo de productos
    df_ordenes = pd.read_csv(ruta_archivo, sep=',')
    
    # 2. Selección de columnas necesarias
    cols_necesarias = [
        'orden_id', 'cliente_id', 'fecha_pedido', 'fecha_entrega',
        'fecha_proceso', 'canal_venta', 'estado', 'monto_total'
    ]
    df_ordenes = df_ordenes[cols_necesarias].copy()
    
    # 3. Transformar fechas
    df_ordenes['fecha_pedido']=pd.to_datetime(df_ordenes['fecha_pedido'], errors='coerce', format='%Y-%m-%d')
    df_ordenes['fecha_entrega']=pd.to_datetime(df_ordenes['fecha_entrega'], errors='coerce', format='%Y-%m-%d')
    df_ordenes['fecha_proceso']=pd.to_datetime(df_ordenes['fecha_proceso'], errors='coerce', format='%Y-%m-%d')
    
    # --- LÓGICA DE METADATOS EN JSON ---
    nombre_archivo = os.path.basename(ruta_archivo)  # Extrae el nombre del archivo de la ruta
    numero_filas = len(df_ordenes)                  # Cuenta el total de filas del DataFrame de productos
    fecha_carga_str = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    
    diccionario_metadatos = {
        "nombre_archivo": nombre_archivo,
        "numero_filas": numero_filas,
        "fecha_carga": fecha_carga_str
    }
    
    # Convierte el diccionario a un String JSON válido
    df_ordenes['metadatos_json'] = json.dumps(diccionario_metadatos)
    # -------------------------------------
    
    # 4. Columnas de auditoría estandarizadas en inglés (coincidentes con tu modelo)
    df_ordenes['source_file'] = nombre_archivo
    df_ordenes['load_date'] = pd.Timestamp.now()
    df_ordenes['update_date'] = pd.Timestamp.now()
    df_ordenes['loaded_by'] = 'proceso_etl_python'
    df_ordenes['is_active'] = 1
    
    return df_ordenes


# --- Ejecución de la función ---
url_ordenes = r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\2.processed\ordenes_cleaned.csv'
df_ordenes = procesar_ordenes(url_ordenes)

# 1. Cargar variables de entorno del archivo .env
load_dotenv()
server = os.getenv('DB_SERVER')
database = os.getenv('DB_DATABASE')

# 2. Validar que las variables existan
if not server or not database:
    raise ValueError("❌ No se encontraron las variables de entorno DB_SERVER o DB_DATABASE.")

print(f"🔄 Conectando al servidor '{server}' en la base de datos '{database}'...")

# 3. Crear la cadena de conexión para SQL Server (Autenticación de Windows)
# Usamos el Driver 17, que es el estándar más compatible
connection_string = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"

# Crear el motor de conexión (Engine) de SQLAlchemy
engine = create_engine(connection_string)

try:    
    print(f"🚀 Iniciando la carga de {len(df_ordenes)} filas a la tabla 'cliente'...")
    
    # 4. Subir a SQL Server en bloques
    df_ordenes.to_sql(
        name='ordenes',            # Nombre exacto de la tabla en SQL Server
        con=engine,                # Nuestro motor de conexión
        if_exists='append',        # 'append' agrega los datos (no borra la tabla existente)
        index=False,               # No sube el índice de Pandas como una columna
        chunksize=1000,            # Manda los datos en bloques de 1,000 en 1,000 filas
    )
    
    print("✅ ¡Carga completada con éxito en bloques de 1000!")

except Exception as e:
    print(f"❌ Ocurrió un error al subir los datos: {e}")

🔄 Conectando al servidor 'DESKTOP-BLERSIO' en la base de datos 'backus'...
🚀 Iniciando la carga de 5602 filas a la tabla 'cliente'...
✅ ¡Carga completada con éxito en bloques de 1000!


# **Load Detalles de Ordenes**

In [59]:
url_detalleordenes=r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\2.processed\detalle_ordenes_cleaned.csv'
df_detalleordenes=pd.read_csv(url_detalleordenes,sep=',')
df_detalleordenes['orden_id'].value_counts()
df_detalleordenes['new_key'].value_counts()
df_detalleordenes.columns

Index(['orden_id', 'linea', 'producto_id', 'cantidad_unidades',
       'precio_unitario', 'descuento_pct', 'new_key', 'es_duplicado',
       'new_key_original'],
      dtype='str')

In [62]:
df_detalleordenes.shape

(19145, 12)

In [60]:
df_detalleordenes.info()

<class 'pandas.DataFrame'>
RangeIndex: 19145 entries, 0 to 19144
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   orden_id           19145 non-null  str    
 1   linea              19145 non-null  int64  
 2   producto_id        19145 non-null  str    
 3   cantidad_unidades  19145 non-null  int64  
 4   precio_unitario    19145 non-null  float64
 5   descuento_pct      19145 non-null  int64  
 6   new_key            19145 non-null  str    
 7   es_duplicado       19145 non-null  bool   
 8   new_key_original   0 non-null      float64
dtypes: bool(1), float64(2), int64(3), str(3)
memory usage: 1.2 MB


In [64]:
def procesar_detalleordenes(ruta_archivo):
    # 1. Carga del archivo de productos
    df_detalleordenes = pd.read_csv(ruta_archivo, sep=',')
    
    # 2. Selección de columnas necesarias
    cols_necesarias = [
        'orden_id', 'linea', 'producto_id', 'cantidad_unidades',
       'precio_unitario', 'descuento_pct'
    ]
    df_detalleordenes = df_detalleordenes[cols_necesarias].copy()
 
    # --- LÓGICA DE METADATOS EN JSON ---
    nombre_archivo = os.path.basename(ruta_archivo)  # Extrae el nombre del archivo de la ruta
    numero_filas = len(df_detalleordenes)                  # Cuenta el total de filas del DataFrame de productos
    fecha_carga_str = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    
    diccionario_metadatos = {
        "nombre_archivo": nombre_archivo,
        "numero_filas": numero_filas,
        "fecha_carga": fecha_carga_str
    }
    
    # Convierte el diccionario a un String JSON válido
    df_detalleordenes['metadatos_json'] = json.dumps(diccionario_metadatos)
    # -------------------------------------
    
    # 4. Columnas de auditoría estandarizadas en inglés (coincidentes con tu modelo)
    df_detalleordenes['source_file'] = nombre_archivo
    df_detalleordenes['load_date'] = pd.Timestamp.now()
    df_detalleordenes['update_date'] = pd.Timestamp.now()
    df_detalleordenes['loaded_by'] = 'proceso_etl_python'
    df_detalleordenes['is_active'] = 1
    
    return df_detalleordenes

url_detalleordenes=r'D:\Github-Time\Solutions-to-Coding-Challenges\Data_Backus\data\2.processed\detalle_ordenes_cleaned.csv'
df_detalleordenes=procesar_detalleordenes(url_detalleordenes)

# 1. Cargar variables de entorno del archivo .env
load_dotenv()
server = os.getenv('DB_SERVER')
database = os.getenv('DB_DATABASE')

# 2. Validar que las variables existan
if not server or not database:
    raise ValueError("❌ No se encontraron las variables de entorno DB_SERVER o DB_DATABASE.")

print(f"🔄 Conectando al servidor '{server}' en la base de datos '{database}'...")

# 3. Crear la cadena de conexión para SQL Server (Autenticación de Windows)
# Usamos el Driver 17, que es el estándar más compatible
connection_string = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"

# Crear el motor de conexión (Engine) de SQLAlchemy
engine = create_engine(connection_string)

try:    
    print(f"🚀 Iniciando la carga de {len(df_ordenes)} filas a la tabla 'cliente'...")
    
    # 4. Subir a SQL Server en bloques
    df_detalleordenes.to_sql(
        name='detalle_ordenes',            # Nombre exacto de la tabla en SQL Server
        con=engine,                # Nuestro motor de conexión
        if_exists='append',        # 'append' agrega los datos (no borra la tabla existente)
        index=False,               # No sube el índice de Pandas como una columna
        chunksize=1000,            # Manda los datos en bloques de 1,000 en 1,000 filas
    )
    
    print("✅ ¡Carga completada con éxito en bloques de 1000!")

except Exception as e:
    print(f"❌ Ocurrió un error al subir los datos: {e}")

🔄 Conectando al servidor 'DESKTOP-BLERSIO' en la base de datos 'backus'...
🚀 Iniciando la carga de 5602 filas a la tabla 'cliente'...
✅ ¡Carga completada con éxito en bloques de 1000!
